# Preprocessing Dataset — Prediksi Ketepatan Lulus Mahasiswa

Notebook ini mendokumentasikan seluruh pipeline preprocessing untuk proyek prediksi ketepatan lulus mahasiswa. Dataset mentah (`dataset.csv`) berisi 608 baris dan 27 kolom dengan berbagai masalah kualitas data: missing values sistematik, system zeros pada nilai IPS, fitur leakage, dan korelasi redundan.

Pipeline ini mengubah dataset mentah menjadi `dataset_clean.csv` yang siap digunakan untuk pemodelan Decision Tree.


## 0. Setup & Load

Kita mulai dengan mengimpor library yang diperlukan, mengatur opsi tampilan pandas, dan memuat dataset mentah.


In [25]:
import pandas as pd
import numpy as np

try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except:
    pass
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

df = pd.read_csv('dataset.csv')
print(f'Shape: {df.shape}')
df.info()
print()
df.head()


## 1. Replace System Zeros (IPS = 0.0 → NaN)

**Temuan EDA:** 36% mahasiswa (219 dari 608) memiliki minimal satu nilai IPS = 0.0, namun 79% di antaranya tetap lulus tepat waktu. Ini membuktikan bahwa nilai 0.0 bukanlah nilai riil, melainkan **placeholder dari sistem legacy**. Jika tidak diperbaiki, nilai ini akan merusak imputasi dan menghasilkan korelasi palsu (ips_sem1 r = -0.27 dengan target — negatif karena 0.0 dianggap rendah).

**Tindakan:** Ganti semua 0.0 dengan NaN pada kolom `ips_sem1-4` dan `sks_sem1-4`.


In [26]:
ips_cols = ['ips_sem1', 'ips_sem2', 'ips_sem3', 'ips_sem4']
sks_cols = ['sks_sem1', 'sks_sem2', 'sks_sem3', 'sks_sem4']

print('Jumlah nilai 0.0 sebelum replacement:')
for col in ips_cols + sks_cols:
    zeros = (df[col] == 0.0).sum()
    print(f'  {col}: {zeros} zeros')

for col in ips_cols + sks_cols:
    df.loc[df[col] == 0.0, col] = np.nan

print('\nJumlah nilai 0.0 setelah replacement:')
for col in ips_cols + sks_cols:
    zeros = (df[col] == 0.0).sum()
    print(f'  {col}: {zeros} zeros')
print('Semua system zero berhasil diganti NaN.')


## 3. Drop Redundant & Leakage Features

Berdasarkan analisis korelasi dan domain knowledge, beberapa fitur harus di-drop:

| Fitur | Alasan | r dengan Target |
|-------|--------|:---:|
| `student_id` | Identitas unik, bukan prediktor | — |
| `ips_sem4` | **Data leakage** — mahasiswa lulus tepat waktu otomatis punya IPS sem4 tinggi | **+0.877** |
| `sks_sem4` | Terkait ips_sem4, leakage | -0.356 |
| `ipk_sem4` | Redundan dengan avg_ips (inter-korelasi > 0.8) | +0.633 |
| `semester_count` | **Circular** — target didefinisikan dari durasi studi | — |
| `ips_max` | Tidak mendiskriminasi kedua kelas (mean ≈ 3.65 untuk kedua kelas) | ≈ 0.000 |
| `total_sks_lulus_sem4` | Redundan dengan `sks_completion_ratio` | — |
| `avg_attendance` | 53% missing + korelasi hampir nol | -0.016 |
| `id_agama` | r=0.031 dengan target, near-zero correlation, hanya 4 mahasiswa Hindu | +0.031 |
| `jenis_kelamin` | r=0.050 dengan target, tidak ada class separation | +0.050 |
| `has_attendance` | Weak signal (4.7% diff), 53% missing | — |

> **Catatan:** Fitur demografis (`id_agama`, `jenis_kelamin`) dan turunan kehadiran (`has_attendance`) di-drop karena korelasi sangat lemah dengan target dan tidak memberikan informasi diskriminatif. Keputusan ini juga mengurangi risiko overfitting pada fitur dengan variabilitas rendah.


In [27]:
drop_cols = [
    'student_id',
    'ips_sem4',
    'sks_sem4',
    'ipk_sem4',
    'semester_count',
    'ips_max',
    'total_sks_lulus_sem4',
    'avg_attendance',
    'id_agama',
    'jenis_kelamin'
]

print('Drop kolom:')
for col in drop_cols:
    print(f'  - {col}')

df.drop(columns=drop_cols, inplace=True)
print(f'\nSisa kolom ({len(df.columns)}): {list(df.columns)}')


## 4. Missing Value Check (Pre-Imputation)

Sebelum imputasi, mari kita lihat distribusi missing value yang tersisa.

**Temuan EDA:** Missing bersifat sistematik, bukan random. Angkatan tua lebih parah:
- Angkatan 2015: rata-rata 1.99 missing per mahasiswa
- Angkatan 2018: rata-rata 1.85 missing
- Angkatan 2022: hanya 0.03 missing

Ini mengonfirmasi bahwa imputasi harus dilakukan per angkatan, bukan secara global.


In [28]:
print('Missing values per kolom (pre-imputation):')
missing = df.isnull().sum()
print(missing[missing > 0].to_string())

print('\nRata-rata missing per angkatan:')
ips_sks_cols = ['ips_sem1', 'ips_sem2', 'ips_sem3', 'sks_sem1', 'sks_sem2', 'sks_sem3']
missing_by_angkatan = df.groupby('angkatan')[ips_sks_cols].apply(
    lambda x: x.isnull().sum(axis=1).mean()
)
print(missing_by_angkatan.to_string())


## 5. Median Imputation per Angkatan

**Mengapa median?** Median robust terhadap outlier, berbeda dengan mean yang bisa terdistorsi oleh nilai IPS ekstrem.

**Mengapa per angkatan?** Pola missing sangat berbeda antar kohort — angkatan tua kehilangan lebih banyak data semester awal karena sistem pencatatan yang belum terkomputerisasi.

**Catatan:** Data hanya 608 baris, sehingga metode kompleks seperti KNN Imputer atau MICE bersifat overkill. Imputasi median per grup sudah memadai untuk Decision Tree.


In [29]:
impute_cols = ['ips_sem1', 'ips_sem2', 'ips_sem3', 'sks_sem1', 'sks_sem2', 'sks_sem3']

print('Imputasi per kolom:')
for col in impute_cols:
    n_before = df[col].isnull().sum()
    df[col] = df.groupby('angkatan')[col].transform(lambda x: x.fillna(x.median()))
    n_after = df[col].isnull().sum()
    print(f'  {col}: {n_before} nulls -> diimputasi -> {n_after} nulls tersisa')

remaining = df[impute_cols].isnull().sum().sum()
print(f'\nTotal nulls tersisa di kolom imputasi: {remaining}')

# Final null check
total_nulls = df.isnull().sum().sum()
print(f'\nVerifikasi final semua nulls: {total_nulls}')
if total_nulls > 0:
    print(df.isnull().sum()[df.isnull().sum() > 0])
else:
    print('  Tidak ada nulls tersisa!')


## 6. Recompute Derived Features

Setelah imputasi, semua derived features (yang sebelumnya bergantung pada nilai IPS/SKS) harus dihitung ulang dari nilai yang sudah bersih:

| Fitur | Formula | Deskripsi |
|-------|---------|-----------|
| `ips_trend` | `ips_sem3 - ips_sem1` | Tren akademik (positif = membaik) |
| `avg_ips` | `mean(ips_sem1, ips_sem2, ips_sem3)` | Rata-rata semester 1-3 |
| `ips_std` | `std(ips_sem1, ips_sem2, ips_sem3)` | Volatilitas kinerja |
| `ips_min` | `min(ips_sem1, ips_sem2, ips_sem3)` | Nilai IPS terendah |
| `sks_completion_ratio` | `(sks_sem1 + sks_sem2 + sks_sem3) / 60` | Proporsi SKS terhadap target 60 SKS |


In [30]:
# ips_trend: last valid minus first valid
ips_have = df[['ips_sem1', 'ips_sem2', 'ips_sem3']]
df['ips_trend'] = ips_have.bfill(axis=1).iloc[:, -1] - ips_have.ffill(axis=1).iloc[:, 0]

# avg_ips
df['avg_ips'] = df[['ips_sem1', 'ips_sem2', 'ips_sem3']].mean(axis=1)

# ips_std
df['ips_std'] = df[['ips_sem1', 'ips_sem2', 'ips_sem3']].std(axis=1)

# ips_min
df['ips_min'] = df[['ips_sem1', 'ips_sem2', 'ips_sem3']].min(axis=1)

# sks_completion_ratio
df['sks_completion_ratio'] = (df['sks_sem1'] + df['sks_sem2'] + df['sks_sem3']) / 60

print('Statistik derived features:')
print(f'  ips_trend           : min={df.ips_trend.min():.4f}, max={df.ips_trend.max():.4f}, mean={df.ips_trend.mean():.4f}')
print(f'  avg_ips             : min={df.avg_ips.min():.4f}, max={df.avg_ips.max():.4f}, mean={df.avg_ips.mean():.4f}')
print(f'  ips_std             : min={df.ips_std.min():.4f}, max={df.ips_std.max():.4f}, mean={df.ips_std.mean():.4f}')
print(f'  ips_min             : min={df.ips_min.min():.4f}, max={df.ips_min.max():.4f}, mean={df.ips_min.mean():.4f}')
print(f'  sks_completion_ratio: min={df.sks_completion_ratio.min():.4f}, max={df.sks_completion_ratio.max():.4f}, mean={df.sks_completion_ratio.mean():.4f}')


## 7. Encode Categorical Features

**Temuan EDA:** Program IH hampir 2x lebih berisiko tidak tepat waktu dibanding AP (12.6% vs 6.8%). Decision Tree di sklearn membutuhkan input numerik, jadi fitur kategorikal harus di-encode.

**Encoding:**
- `program`: AP → 0, IH → 1


In [31]:
df['program'] = df['program'].map({'AP': 0, 'IH': 1})

print('Unique values setelah encoding:')
print(f'  program       : {sorted(df.program.unique())}')


## 8. Final Dataset Summary

Setelah seluruh pipeline preprocessing selesai, mari kita reorder kolom, lalu verifikasi dataset final sebelum disimpan.


In [32]:
# Reorder columns to match expected output
expected_order = [
    'angkatan', 'program',
    'ips_sem1', 'ips_sem2', 'ips_sem3',
    'sks_sem1', 'sks_sem2', 'sks_sem3',
    'failed_courses', 'failed_in_sem1', 'repeated_courses',
    'ips_trend', 'avg_ips', 'ips_std', 'ips_min', 'sks_completion_ratio',
    'target'
]
actual_order = [c for c in expected_order if c in df.columns]
df = df[actual_order]

print(f'Final shape: {df.shape}')
print(f'Total NULLs: {df.isnull().sum().sum()}')
print(f'\nInfo:')
df.info()
print(f'\nDeskripsi statistik:')
df.describe()
print(f'\nDistribusi target:')
target_counts = df.target.value_counts()
target_pct = df.target.mean() * 100
print(f'  {target_counts.to_dict()}')
print(f'  Tepat waktu: {target_pct:.2f}%')
print(f'  Tidak tepat: {100 - target_pct:.2f}%')
print(f'\n5 baris pertama:')
df.head()


## 9. Save Clean Dataset

Simpan dataset yang sudah bersih ke `dataset_clean.csv`. Dataset final memiliki 608 baris, 16 fitur + 1 target (total 17 kolom), tanpa NULL.


In [24]:
df.to_csv('dataset_clean_nb.csv', index=False)
print(f'Dataset berhasil disimpan: dataset_clean.csv ({df.shape[0]} rows x {df.shape[1]} cols)')
print('Pipeline preprocessing selesai.')
